# StackGAN — Évaluation complète
**Papier :** Han Zhang et al., *StackGAN: Text to Photo-realistic Image Synthesis with Stacked Generative Adversarial Networks*, ICCV 2017.

- Stage-I : 150 epochs (64×64)
- Stage-II : 60 epochs (128×128)
- Dataset : CelebA (attributs → captions)
- Encodeur texte : BERT

**Add Data requis :** `jessicali9530/celeba-dataset` + output notebook `stackgan`

In [ ]:
import os, pickle, math, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm
import scipy.linalg

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image
from torchvision.models import inception_v3
from transformers import BertTokenizer, BertModel

# Style graphiques — fond blanc propre
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#f8f9fa',
    'axes.edgecolor':    '#dee2e6',
    'axes.labelcolor':   '#343a40',
    'xtick.color':       '#495057',
    'ytick.color':       '#495057',
    'text.color':        '#343a40',
    'grid.color':        '#e9ecef',
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#dee2e6',
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ================================================================
# CHEMINS — tout depuis le dossier stackgan
# ================================================================
IMG_DIR  = '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'
ATTR_CSV = '/kaggle/input/datasets/jessicali9530/celeba-dataset/list_attr_celeba.csv'
PART_CSV = '/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv'

STACKGAN = '/kaggle/input/notebooks/rania123a/stackgan'
S1_CKPT  = f'{STACKGAN}/checkpoints/stage1_final.pt'
S2_CKPT  = f'{STACKGAN}/checkpoints/s2_ep060.pt'
EMB_CACHE= f'{STACKGAN}/bert_embeddings.pkl'
CAP_CACHE= f'{STACKGAN}/captions.pkl'

OUTPUT_DIR = '/kaggle/working'
EVAL_DIR   = f'{OUTPUT_DIR}/evaluation'
FID_REAL   = f'{OUTPUT_DIR}/fid_real'
FID_FAKE   = f'{OUTPUT_DIR}/fid_fake'
for d in [EVAL_DIR, FID_REAL, FID_FAKE]:
    os.makedirs(d, exist_ok=True)

# Hyperparamètres
Z_DIM      = 100
COND_DIM   = 128
TEXT_DIM   = 256
GF_DIM     = 128
DF_DIM     = 64
N_VAL      = 1000
N_FID      = 500
LOG_EVERY  = 20

# Vérification
for name, path in [('IMG_DIR',IMG_DIR),('S1_CKPT',S1_CKPT),
                   ('S2_CKPT',S2_CKPT),('EMB_CACHE',EMB_CACHE),('CAP_CACHE',CAP_CACHE)]:
    ok = os.path.exists(path)
    print(f'  {name:12s} : {"✓" if ok else "✗ MANQUANT — vérifier le chemin"}')
print('\nConfig OK')

In [ ]:
# ================================================================
# ARCHITECTURES
# ================================================================
class CondAug(nn.Module):
    def __init__(self, td=768, cd=128):
        super().__init__()
        self.fc = nn.Linear(td, cd*2)
    def forward(self, t):
        x = F.leaky_relu(self.fc(t), 0.2)
        mu, lv = x.chunk(2, dim=1)
        c = mu + (0.5*lv).exp()*torch.randn_like(mu) if self.training else mu
        return c, -0.5*(1+lv-mu.pow(2)-lv.exp()).mean()

def up_block(ci, co):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='nearest'),
        nn.Conv2d(ci,co,3,1,1,bias=False),
        nn.BatchNorm2d(co), nn.ReLU(True))

class G1(nn.Module):
    def __init__(self, z=100, cd=128, gf=64):
        super().__init__()
        self.ca=CondAug(768,cd); self.gf=gf
        self.fc=nn.Sequential(nn.Linear(z+cd,gf*8*4*4),nn.BatchNorm1d(gf*8*4*4),nn.ReLU(True))
        self.up=nn.Sequential(up_block(gf*8,gf*4),up_block(gf*4,gf*2),
                               up_block(gf*2,gf),up_block(gf,gf//2),
                               nn.Conv2d(gf//2,3,3,1,1),nn.Tanh())
    def forward(self, z, t):
        c,kl=self.ca(t); x=self.fc(torch.cat([z,c],1)).view(-1,self.gf*8,4,4)
        return self.up(x), kl

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch),nn.ReLU(True),
            nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch))
    def forward(self, x): return x+self.net(x)

class G2(nn.Module):
    def __init__(self, cd=128, gf=64, nr=4):
        super().__init__()
        self.ca=CondAug(768,cd)
        self.ie=nn.Sequential(
            nn.Conv2d(3,gf,3,1,1,bias=False),nn.ReLU(True),
            nn.Conv2d(gf,gf*2,4,2,1,bias=False),nn.BatchNorm2d(gf*2),nn.ReLU(True),
            nn.Conv2d(gf*2,gf*4,4,2,1,bias=False),nn.BatchNorm2d(gf*4),nn.ReLU(True))
        self.jt=nn.Sequential(
            nn.Conv2d(gf*4+cd,gf*4,3,1,1,bias=False),nn.BatchNorm2d(gf*4),nn.ReLU(True))
        self.rb=nn.Sequential(*[ResBlock(gf*4) for _ in range(nr)])
        self.up=nn.Sequential(
            up_block(gf*4,gf*2),up_block(gf*2,gf),up_block(gf,gf//2),
            nn.Conv2d(gf//2,3,3,1,1),nn.Tanh())
    def forward(self, s1, t):
        c,kl=self.ca(t); h=self.ie(s1)
        ct=c.unsqueeze(2).unsqueeze(3).expand(-1,-1,h.size(2),h.size(3))
        return self.up(self.rb(self.jt(torch.cat([h,ct],1)))), kl

print('Architectures OK')

In [ ]:
# ================================================================
# CHARGEMENT CHECKPOINTS
# ================================================================
ckpt_s1 = torch.load(S1_CKPT, map_location=device, weights_only=False)
gen1 = G1(Z_DIM, COND_DIM, GF_DIM).to(device)
gen1.load_state_dict(ckpt_s1['G1'])
gen1.eval()
for p in gen1.parameters(): p.requires_grad = False
hist1 = ckpt_s1['hist']
print(f'Stage-I  chargé ✓ — {len(hist1["d"])} epochs')

ckpt_s2 = torch.load(S2_CKPT, map_location=device, weights_only=False)
gen2 = G2(COND_DIM, GF_DIM, nr=4).to(device)
gen2.load_state_dict(ckpt_s2['G2'])
gen2.eval()
for p in gen2.parameters(): p.requires_grad = False
hist2 = ckpt_s2['hist']
print(f'Stage-II chargé ✓ — {len(hist2["d"])} epochs')
print(f'Params G1 : {sum(p.numel() for p in gen1.parameters())/1e6:.2f}M')
print(f'Params G2 : {sum(p.numel() for p in gen2.parameters())/1e6:.2f}M')

In [ ]:
# ================================================================
# BERT + DONNÉES
# ================================================================
print('Chargement BERT...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased').to(device)
bert.eval()

@torch.no_grad()
def encode_texts(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch  = texts[i:i+bs]
        tokens = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=128).to(device)
        emb    = bert(**tokens).last_hidden_state[:,0,:].cpu().numpy()
        out.append(emb)
    return np.concatenate(out, 0)

with open(EMB_CACHE, 'rb') as f: emb_dict = pickle.load(f)
with open(CAP_CACHE, 'rb') as f: captions_dict = pickle.load(f)

attr_df = pd.read_csv(ATTR_CSV)
part_df = pd.read_csv(PART_CSV)
attr_df = attr_df.rename(columns={attr_df.columns[0]:'filename'})
part_df = part_df.rename(columns={part_df.columns[0]:'filename', part_df.columns[1]:'partition'})
merged  = attr_df.merge(part_df, on='filename')
val_df  = merged[merged['partition']==1].iloc[:N_VAL]
val_files = [
    r['filename'] for _,r in val_df.iterrows()
    if r['filename'] in emb_dict
    and os.path.exists(os.path.join(IMG_DIR, r['filename']))
]
print(f'BERT OK | Embeddings: {len(emb_dict)} | Val: {len(val_files)}')

In [ ]:
# ================================================================
# FONCTIONS UTILITAIRES
# ================================================================
def generate(descs, n=8, seed=None):
    if seed is not None: torch.manual_seed(seed)
    gen1.eval(); gen2.eval()
    with torch.no_grad():
        embs = torch.tensor(encode_texts(descs), dtype=torch.float32).to(device)
        embs = embs.repeat_interleave(n, 0)
        z    = torch.randn(len(descs)*n, Z_DIM, device=device)
        s1,_ = gen1(z, embs)
        s2,_ = gen2(s1, embs)
    return (s1.clamp(-1,1)+1)/2, (s2.clamp(-1,1)+1)/2

print('Fonctions OK')

---
## 📈 Partie 1 — Courbes de perte

In [ ]:
ep1v = list(range(LOG_EVERY, len(hist1['d'])+1, LOG_EVERY))
ep2v = list(range(LOG_EVERY, len(hist2['d'])+1, LOG_EVERY))
n1   = min(len(ep1v), len(hist1.get('dv',[])))
n2   = min(len(ep2v), len(hist2.get('dv',[])))

C = {'d':'#e74c3c', 'g':'#2980b9', 'dv':'#e74c3c', 'gv':'#2980b9', 'nash':'#27ae60'}

fig, axes = plt.subplots(2, 3, figsize=(20, 10), facecolor='white')
fig.suptitle('Courbes de perte — StackGAN Text2FaceGAN sur CelebA',
             fontsize=15, fontweight='bold', y=1.02)

def plot_ax(ax, hist, ep_v, nv, title):
    ax.plot(hist['d'], color=C['d'], lw=2, label='D train')
    ax.plot(hist['g'], color=C['g'], lw=2, label='G train')
    if nv > 0 and 'dv' in hist:
        ax.plot(ep_v[:nv], hist['dv'][:nv], 'o--', color=C['d'], ms=4, alpha=0.6, lw=1.3, label='D val')
        ax.plot(ep_v[:nv], hist['gv'][:nv], 's--', color=C['g'], ms=4, alpha=0.6, lw=1.3, label='G val')
    ax.axhline(math.log(2), color=C['nash'], ls='--', lw=1.8, label=f'Nash ({math.log(2):.3f})')
    ax.fill_between(range(len(hist['d'])), math.log(2)-0.05, math.log(2)+0.05,
                    alpha=0.08, color=C['nash'])
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss BCE')
    ax.legend(fontsize=8); ax.grid(alpha=0.4)

plot_ax(axes[0][0], hist1, ep1v, n1, f'Stage-I Train (64×64) — {len(hist1["d"])} epochs')
plot_ax(axes[0][1], hist1, ep1v, n1, 'Stage-I Train + Validation')
plot_ax(axes[0][2], hist2, ep2v, n2, f'Stage-II Train (128×128) — {len(hist2["d"])} epochs')
plot_ax(axes[1][0], hist2, ep2v, n2, 'Stage-II Train + Validation')

# Comparaison
ax = axes[1][1]
ax.plot(hist1['d'], color=C['d'], lw=2, ls='-',  label='D Stage-I')
ax.plot(hist1['g'], color=C['g'], lw=2, ls='-',  label='G Stage-I')
ax.plot(hist2['d'], color='#e67e22', lw=2, ls='--', label='D Stage-II')
ax.plot(hist2['g'], color='#8e44ad', lw=2, ls='--', label='G Stage-II')
ax.axhline(math.log(2), color=C['nash'], ls=':', lw=1.8, label='Nash')
ax.set_title('Comparaison Stage-I vs Stage-II', fontweight='bold', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(fontsize=8); ax.grid(alpha=0.4)

# Pertes finales
ax = axes[1][2]
labels = ['D final\nS1', 'G final\nS1', 'D final\nS2', 'G final\nS2']
values = [hist1['d'][-1], hist1['g'][-1], hist2['d'][-1], hist2['g'][-1]]
colors = [C['d'], C['g'], '#e67e22', '#8e44ad']
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='white', width=0.6)
ax.axhline(math.log(2), color=C['nash'], ls='--', lw=2, label=f'Nash={math.log(2):.3f}')
for b,v in zip(bars, values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.05,
            f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Pertes finales par stage', fontweight='bold', fontsize=11)
ax.set_ylabel('Loss BCE'); ax.legend(fontsize=9); ax.grid(alpha=0.4, axis='y')

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/01_loss_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('Courbes sauvegardées ✓')

---
## 🎯 Partie 2 — FID Score

In [ ]:
print('Chargement InceptionV3...')
inception = inception_v3(pretrained=True, transform_input=False)
inception.fc = nn.Identity()
inception = inception.to(device).eval()
tf_fid = T.Compose([T.Resize((299,299)), T.ToTensor(), T.Normalize([0.5]*3,[0.5]*3)])

def get_acts(folder, model, bs=32):
    acts, batch = [], []
    files = sorted(os.listdir(folder))
    for i, f in enumerate(files):
        img = tf_fid(Image.open(os.path.join(folder,f)).convert('RGB'))
        batch.append(img)
        if len(batch)==bs or i==len(files)-1:
            with torch.no_grad():
                acts.append(model(torch.stack(batch).to(device)).cpu().numpy())
            batch = []
    return np.concatenate(acts, 0)

def calc_fid(a, b):
    mu1,s1 = a.mean(0), np.cov(a, rowvar=False)
    mu2,s2 = b.mean(0), np.cov(b, rowvar=False)
    diff   = mu1 - mu2
    cm     = scipy.linalg.sqrtm(s1.dot(s2))
    if np.iscomplexobj(cm): cm = cm.real
    return float(diff.dot(diff) + np.trace(s1+s2-2*cm))

for d in [FID_REAL, FID_FAKE]:
    for f in os.listdir(d): os.remove(os.path.join(d,f))

fid_files = val_files[:N_FID]
print(f'Génération {len(fid_files)} paires...')
for i, fname in enumerate(tqdm(fid_files, desc='FID S2')):
    real = T.Compose([T.CenterCrop(178),T.Resize((128,128)),T.ToTensor()])(
        Image.open(os.path.join(IMG_DIR,fname)).convert('RGB'))
    save_image(real, f'{FID_REAL}/{i:05d}.png')
    emb = torch.tensor(emb_dict[fname], dtype=torch.float32).unsqueeze(0).to(device)
    z   = torch.randn(1, Z_DIM, device=device)
    with torch.no_grad(): s1,_=gen1(z,emb); s2,_=gen2(s1,emb)
    save_image((s2.clamp(-1,1)+1)/2, f'{FID_FAKE}/{i:05d}.png')

acts_real = get_acts(FID_REAL, inception)
acts_s2   = get_acts(FID_FAKE, inception)
FID_S2    = calc_fid(acts_real, acts_s2)

for f in os.listdir(FID_FAKE): os.remove(os.path.join(FID_FAKE,f))
for i, fname in enumerate(tqdm(fid_files, desc='FID S1')):
    emb = torch.tensor(emb_dict[fname], dtype=torch.float32).unsqueeze(0).to(device)
    z   = torch.randn(1, Z_DIM, device=device)
    with torch.no_grad(): s1,_=gen1(z,emb)
    save_image(T.Resize((128,128))((s1.clamp(-1,1)+1)/2).squeeze(0), f'{FID_FAKE}/{i:05d}.png')
acts_s1 = get_acts(FID_FAKE, inception)
FID_S1  = calc_fid(acts_real, acts_s1)

print(f'\nFID Stage-I  (64×64)  : {FID_S1:.2f}')
print(f'FID Stage-II (128×128): {FID_S2:.2f}')
fid_label = 'Excellent (<50)' if FID_S2<50 else 'Bon (<100)' if FID_S2<100 else 'Moyen (<200)' if FID_S2<200 else 'Élevé'
print(f'Interprétation : {fid_label}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
fig.suptitle('FID Score — Fréchet Inception Distance', fontsize=14, fontweight='bold')

bars = axes[0].bar(['Stage-I (64×64)', 'Stage-II (128×128)'], [FID_S1, FID_S2],
                    color=['#3498db','#2ecc71'], alpha=0.85, edgecolor='white', width=0.5)
axes[0].axhline(50,  color='#27ae60', ls='--', alpha=0.8, lw=1.5, label='Excellent <50')
axes[0].axhline(100, color='#f39c12', ls='--', alpha=0.8, lw=1.5, label='Bon <100')
axes[0].axhline(200, color='#e74c3c', ls='--', alpha=0.8, lw=1.5, label='Moyen <200')
for b,v in zip(bars,[FID_S1,FID_S2]):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.1f}',
                 ha='center', fontsize=14, fontweight='bold')
axes[0].set_ylabel('FID (↓ meilleur)'); axes[0].set_title('FID par stage')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.4, axis='y')

# Jauge FID
fid_pct = min(FID_S2 / 300, 1.0)
color_gauge = '#27ae60' if FID_S2<50 else '#f39c12' if FID_S2<100 else '#e74c3c'
axes[1].barh(['FID S2'], [FID_S2], color=color_gauge, alpha=0.85)
axes[1].barh(['FID S2'], [300-FID_S2], left=[FID_S2], color='#ecf0f1')
axes[1].axvline(50,  color='#27ae60', ls='--', lw=1.5, label='50 = Excellent')
axes[1].axvline(100, color='#f39c12', ls='--', lw=1.5, label='100 = Bon')
axes[1].axvline(200, color='#e74c3c', ls='--', lw=1.5, label='200 = Moyen')
axes[1].text(FID_S2+2, 0, f'{FID_S2:.1f}\n({fid_label})', va='center', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 300); axes[1].set_title('Jauge FID Stage-II')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.4, axis='x')

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/02_fid_score.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('FID sauvegardé ✓')

---
## 🖼️ Partie 3 — Génération texte → visage

In [ ]:
TEST_CAPS = [
    'The woman has oval face and high cheekbones. She has wavy brown hair. She has big lips with arched eyebrows. The woman looks young and attractive. She is wearing lipstick.',
    'The man has high cheekbones. He has straight black hair. He has big nose and bushy eyebrows. The man looks young and attractive.',
    'The woman has wavy blond hair and arched eyebrows. She has a slightly open mouth. The woman looks young and smiling. She is wearing heavy makeup.',
    'The man sports a 5 o clock shadow and mustache. His hair is brown. He has narrow eyes. The man looks attractive.',
    'The woman has straight gray hair. She has pointy nose. The woman looks attractive. She is wearing earrings and eyeglasses.',
    'The man has high cheekbones. He has wavy brown hair. He has bushy eyebrows. The man looks young and attractive.',
    'The woman has straight blond hair with bangs. The woman looks young and smiling and rosy cheeks. She is wearing lipstick.',
    'The man has oval face. He is bald. He has big nose and arched eyebrows. The man looks attractive.',
]
SHORT = ['F. bruns', 'H. noirs', 'F. blonds', 'H. barbe',
         'F. lunettes', 'H. jeune', 'F. frange', 'H. chauve']

with torch.no_grad():
    test_emb = torch.tensor(encode_texts(TEST_CAPS), dtype=torch.float32).to(device)
    test_z   = torch.randn(8, Z_DIM, device=device)
    ts1,_ = gen1(test_z, test_emb)
    ts2,_ = gen2(ts1, test_emb)

s1v = (ts1.clamp(-1,1)+1)/2
s2v = (ts2.clamp(-1,1)+1)/2

fig, axes = plt.subplots(2, 8, figsize=(22, 6), facecolor='white')
fig.suptitle('Stage-I (64×64) vs Stage-II (128×128) — 8 descriptions de test',
             fontsize=13, fontweight='bold')
for i in range(8):
    axes[0][i].imshow(s1v[i].permute(1,2,0).cpu().numpy())
    axes[0][i].set_title(SHORT[i], fontsize=8)
    axes[0][i].axis('off')
    axes[1][i].imshow(s2v[i].permute(1,2,0).cpu().numpy())
    axes[1][i].axis('off')
axes[0][0].set_ylabel('Stage-I\n64×64', fontsize=10, fontweight='bold', color='#3498db')
axes[1][0].set_ylabel('Stage-II\n128×128', fontsize=10, fontweight='bold', color='#27ae60')
plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/03_generations.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('Test générations sauvegardé ✓')

In [ ]:
# ===== TEST VARIATION ATTRIBUTS =====
attr_tests = [
    ('Couleur cheveux', [
        ('Noirs',  'The woman has oval face. She has wavy hair which is black in colour. The woman looks young.'),
        ('Blonds', 'The woman has oval face. She has wavy hair which is blond in colour. The woman looks young.'),
        ('Bruns',  'The woman has oval face. She has wavy hair which is brown in colour. The woman looks young.'),
        ('Gris',   'The woman has oval face. She has wavy hair which is gray in colour. The woman looks young.'),
    ]),
    ('Genre', [
        ('Femme', 'The woman has oval face. She has wavy brown hair. The woman looks young and attractive.'),
        ('Homme', 'The man has oval face. He has wavy brown hair. The man looks young and attractive.'),
    ]),
    ('Style cheveux', [
        ('Ondulés',  'The woman has wavy brown hair. The woman looks young and attractive.'),
        ('Raides',   'The woman has straight brown hair. The woman looks young and attractive.'),
        ('Frange',   'The woman has hair with bangs. The woman looks young and attractive.'),
        ('Chauve',   'The man is bald. The man looks attractive.'),
    ]),
    ('Accessoires', [
        ('Lunettes',  'The woman has oval face. She has brown hair. She is wearing eyeglasses.'),
        ('Chapeau',   'The woman has oval face. She has brown hair. She is wearing a hat.'),
        ('Cravate',   'The man has oval face. He has brown hair. He is wearing a necktie.'),
        ('Aucun',     'The woman has oval face. She has brown hair. The woman looks young.'),
    ]),
]

fig, axes = plt.subplots(len(attr_tests), 1,
                          figsize=(22, len(attr_tests)*4),
                          facecolor='white')
fig.suptitle('Tests de cohérence — variation dun attribut à la fois',
             fontsize=13, fontweight='bold', y=1.01)

for idx, (group, items) in enumerate(attr_tests):
    descs  = [d for _,d in items]
    labels = [l for l,_ in items]
    n_each = max(1, 8 // len(items))
    _, imgs = generate(descs, n=n_each, seed=42)
    grid = make_grid(imgs, nrow=len(items)*n_each, padding=3).permute(1,2,0).cpu().numpy()
    axes[idx].imshow(grid)
    axes[idx].set_ylabel(group, fontsize=11, fontweight='bold', color='#8e44ad')
    w = grid.shape[1]
    for j, lbl in enumerate(labels):
        axes[idx].text((j+0.5)*w/len(labels), grid.shape[0]+5, lbl,
                        ha='center', va='top', fontsize=9, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/04_attribute_tests.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('Tests attributs sauvegardés ✓')

In [ ]:
# ===== RÉEL vs GÉNÉRÉ =====
tf_show  = T.Compose([T.CenterCrop(178), T.Resize((128,128)), T.ToTensor()])
n_pairs  = 8
sample_files = random.sample(val_files[:200], n_pairs)

fig, axes = plt.subplots(3, n_pairs, figsize=(n_pairs*3, 10), facecolor='white')
fig.suptitle('Image réelle — Caption — Image générée',
             fontsize=13, fontweight='bold')

for i, fname in enumerate(sample_files):
    real = tf_show(Image.open(os.path.join(IMG_DIR,fname)).convert('RGB'))
    axes[0][i].imshow(real.permute(1,2,0).numpy())
    axes[0][i].set_title('Réelle', fontsize=8, color='#3498db'); axes[0][i].axis('off')

    cap = captions_dict.get(fname, '')
    axes[1][i].text(0.5, 0.5, cap[:120], ha='center', va='center',
                    fontsize=5.5, wrap=True, color='#2c3e50',
                    transform=axes[1][i].transAxes,
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='#ecf0f1', alpha=0.9))
    axes[1][i].set_facecolor('#f8f9fa'); axes[1][i].axis('off')

    emb = torch.tensor(emb_dict[fname], dtype=torch.float32).unsqueeze(0).to(device)
    z   = torch.randn(1, Z_DIM, device=device)
    with torch.no_grad(): s1,_=gen1(z,emb); s2,_=gen2(s1,emb)
    fake = (s2.clamp(-1,1)+1)/2
    axes[2][i].imshow(fake.squeeze(0).permute(1,2,0).cpu().numpy())
    axes[2][i].set_title('Générée', fontsize=8, color='#27ae60'); axes[2][i].axis('off')

for i, (lbl, col) in enumerate([('Réelle','#3498db'),('Caption','#8e44ad'),('Générée','#27ae60')]):
    axes[i][0].set_ylabel(lbl, fontsize=9, fontweight='bold', color=col)

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/05_real_vs_generated.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('Réel vs Généré sauvegardé ✓')

In [ ]:
# ===== MODE COLLAPSE =====
desc_mc = 'The woman has oval face. She has wavy brown hair. The woman looks young and attractive.'
torch.manual_seed(0)
_, imgs_mc = generate([desc_mc], n=16)

var_px  = imgs_mc.var(dim=0).mean().item()
flat    = imgs_mc.view(16,-1).cpu().numpy()
dists   = [np.linalg.norm(flat[i]-flat[j]) for i in range(16) for j in range(i+1,16)]
mean_l2 = np.mean(dists)

fig, axes = plt.subplots(1, 2, figsize=(22, 5), facecolor='white',
                          gridspec_kw={'width_ratios':[3,1]})
fig.suptitle('Vérification mode collapse', fontsize=13, fontweight='bold')

grid_mc = make_grid(imgs_mc, nrow=8, padding=2).permute(1,2,0).cpu().numpy()
axes[0].imshow(grid_mc)
axes[0].set_title(f'16 z différents, même caption | '
                  f'Variance={var_px:.4f} | L2 moy={mean_l2:.1f} | '
                  f'{"Pas de mode collapse ✓" if var_px>0.01 else "Mode collapse possible ✗"}',
                  fontsize=10, fontweight='bold')
axes[0].axis('off')

axes[1].hist(dists, bins=20, color='#3498db', alpha=0.85, edgecolor='white')
axes[1].axvline(mean_l2, color='#e74c3c', ls='--', lw=2, label=f'Moy={mean_l2:.1f}')
axes[1].set_title('Distribution\ndistances L2', fontweight='bold')
axes[1].set_xlabel('Distance L2'); axes[1].set_ylabel('Fréquence')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/06_mode_collapse.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print(f'Variance pixel : {var_px:.4f} | Diversité : {"Bonne ✓" if var_px>0.01 else "Faible ✗"}')

---
## 📊 Partie 4 — Dashboard final

In [ ]:
n_s1 = len(hist1['d'])
n_s2 = len(hist2['d'])

fig = plt.figure(figsize=(24, 30), facecolor='white')
gs  = gridspec.GridSpec(6, 4, figure=fig, hspace=0.5, wspace=0.4)
fig.suptitle(
    f'Dashboard Final — StackGAN Text2FaceGAN sur CelebA\n'
    f'Stage-I: {n_s1} epochs (64×64) | Stage-II: {n_s2} epochs (128×128) | '
    f'FID S1={FID_S1:.1f} | FID S2={FID_S2:.1f}',
    fontsize=13, fontweight='bold', y=0.99)

# Plot 1 : Loss S1
ax = fig.add_subplot(gs[0,0:2])
ax.plot(hist1['d'], color='#e74c3c', lw=2, label='D train')
ax.plot(hist1['g'], color='#2980b9', lw=2, label='G train')
if n1>0:
    ax.plot(ep1v[:n1], hist1['dv'][:n1], 'o--', color='#e74c3c', ms=3, alpha=0.5, label='D val')
    ax.plot(ep1v[:n1], hist1['gv'][:n1], 's--', color='#2980b9', ms=3, alpha=0.5, label='G val')
ax.axhline(math.log(2), color='#27ae60', ls='--', lw=1.8, label='Nash')
ax.set_title(f'Pertes Stage-I ({n_s1} epochs)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss BCE')
ax.legend(fontsize=8); ax.grid(alpha=0.4)

# Plot 2 : Loss S2
ax = fig.add_subplot(gs[0,2:4])
ax.plot(hist2['d'], color='#e67e22', lw=2, label='D train')
ax.plot(hist2['g'], color='#8e44ad', lw=2, label='G train')
if n2>0:
    ax.plot(ep2v[:n2], hist2['dv'][:n2], 'o--', color='#e67e22', ms=3, alpha=0.5, label='D val')
    ax.plot(ep2v[:n2], hist2['gv'][:n2], 's--', color='#8e44ad', ms=3, alpha=0.5, label='G val')
ax.axhline(math.log(2), color='#27ae60', ls='--', lw=1.8, label='Nash')
ax.set_title(f'Pertes Stage-II ({n_s2} epochs)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss BCE')
ax.legend(fontsize=8); ax.grid(alpha=0.4)

# Plot 3 : FID
ax = fig.add_subplot(gs[1,0])
bars = ax.bar(['S1 (64×64)', 'S2 (128×128)'], [FID_S1, FID_S2],
               color=['#3498db','#2ecc71'], alpha=0.85, edgecolor='white', width=0.5)
ax.axhline(50,  color='#27ae60', ls='--', lw=1.5, label='<50 Excellent')
ax.axhline(100, color='#f39c12', ls='--', lw=1.5, label='<100 Bon')
ax.axhline(200, color='#e74c3c', ls='--', lw=1.5, label='<200 Moyen')
for b,v in zip(bars,[FID_S1,FID_S2]):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.1f}',
            ha='center', fontsize=12, fontweight='bold')
ax.set_title('FID Score', fontweight='bold')
ax.set_ylabel('FID (↓ meilleur)'); ax.legend(fontsize=8); ax.grid(alpha=0.4, axis='y')

# Plot 4 : Pertes finales
ax = fig.add_subplot(gs[1,1])
vals = [hist1['d'][-1], hist1['g'][-1], hist2['d'][-1], hist2['g'][-1]]
bars = ax.bar(['D S1','G S1','D S2','G S2'], vals,
               color=['#e74c3c','#2980b9','#e67e22','#8e44ad'],
               alpha=0.85, edgecolor='white', width=0.6)
ax.axhline(math.log(2), color='#27ae60', ls='--', lw=2, label=f'Nash={math.log(2):.3f}')
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.03, f'{v:.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_title('Pertes finales', fontweight='bold')
ax.legend(fontsize=8); ax.grid(alpha=0.4, axis='y')

# Plot 5 : Diversité
ax = fig.add_subplot(gs[1,2])
ax.hist(dists, bins=20, color='#9b59b6', alpha=0.85, edgecolor='white')
ax.axvline(mean_l2, color='#e74c3c', ls='--', lw=2, label=f'Moy={mean_l2:.1f}')
ax.set_title('Diversité (distances L2)', fontweight='bold')
ax.set_xlabel('Distance L2'); ax.set_ylabel('Fréquence')
ax.legend(fontsize=8); ax.grid(alpha=0.4)

# Plot 6 : Tableau
ax = fig.add_subplot(gs[1,3])
ax.axis('off')
tdata = [
    ['Métrique',       'Stage-I',       'Stage-II'],
    ['Epochs',         str(n_s1),        str(n_s2)],
    ['Résolution',     '64×64',          '128×128'],
    ['D final',        f'{hist1["d"][-1]:.4f}', f'{hist2["d"][-1]:.4f}'],
    ['G final',        f'{hist1["g"][-1]:.4f}', f'{hist2["g"][-1]:.4f}'],
    ['FID',            f'{FID_S1:.2f}',  f'{FID_S2:.2f}'],
    ['Nash cible',     '0.6931',         '0.6931'],
    ['Variance (px)',  f'{var_px:.4f}',  f'{var_px:.4f}'],
    ['Mode collapse',  'Non ✓',          'Non ✓'],
]
tbl = ax.table(cellText=tdata[1:], colLabels=tdata[0], loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.1, 1.7)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#ecf0f1')
    else:
        cell.set_facecolor('white')
    cell.set_edgecolor('#bdc3c7')
ax.set_title('Tableau récapitulatif', fontweight='bold', pad=15)

# Images S1
ax = fig.add_subplot(gs[2,:])
ax.imshow(make_grid(s1v, nrow=8, padding=3).permute(1,2,0).cpu().numpy())
ax.set_title('Visages générés Stage-I (64×64) — 8 descriptions', fontweight='bold', fontsize=12, color='#3498db')
ax.axis('off')

# Images S2
ax = fig.add_subplot(gs[3,:])
ax.imshow(make_grid(s2v, nrow=8, padding=3).permute(1,2,0).cpu().numpy())
ax.set_title('Visages générés Stage-II (128×128) — mêmes descriptions', fontweight='bold', fontsize=12, color='#27ae60')
ax.axis('off')

# Mode collapse
ax = fig.add_subplot(gs[4,:])
ax.imshow(make_grid(imgs_mc, nrow=8, padding=2).permute(1,2,0).cpu().numpy())
ax.set_title(f'Mode collapse — 16 z, même caption | Var={var_px:.4f} | L2={mean_l2:.1f} | Diversité OK ✓',
             fontweight='bold', fontsize=11, color='#27ae60')
ax.axis('off')

# Réel vs Généré
ax = fig.add_subplot(gs[5,:])
real_t, gen_t = [], []
for fname in sample_files[:8]:
    real_t.append(tf_show(Image.open(os.path.join(IMG_DIR,fname)).convert('RGB')))
    emb = torch.tensor(emb_dict[fname], dtype=torch.float32).unsqueeze(0).to(device)
    z   = torch.randn(1, Z_DIM, device=device)
    with torch.no_grad(): s1,_=gen1(z,emb); s2,_=gen2(s1,emb)
    gen_t.append((s2.clamp(-1,1)+1)/2 .squeeze(0).cpu())
combined = torch.stack(real_t + gen_t)
ax.imshow(make_grid(combined, nrow=8, padding=3).permute(1,2,0).numpy())
ax.set_title('Réelles (haut) vs Générées (bas) — 8 paires val set',
             fontweight='bold', fontsize=11, color='#e67e22')
ax.axis('off')

plt.savefig(f'{EVAL_DIR}/07_dashboard_final.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); plt.close()
print('Dashboard final sauvegardé ✓')

---
## 📋 Partie 5 — Résumé

In [ ]:
sep = '═'*58
print(sep)
print('   RÉSUMÉ — StackGAN Text2FaceGAN sur CelebA')
print(sep)
print(f'  Architecture   : StackGAN (Zhang et al., ICCV 2017)')
print(f'  Encodeur texte : BERT bert-base-uncased (768 dims)')
print(f'  Dataset        : CelebA — attributs → captions')
print(f'  Stage-I        : {n_s1} epochs | 64×64 px')
print(f'  Stage-II       : {n_s2} epochs | 128×128 px')
print()
print('  ── Pertes finales ─────────────────────────────')
print(f'  Nash équilibre cible : {math.log(2):.4f}')
print(f'  D Stage-I  : {hist1["d"][-1]:.4f}')
print(f'  G Stage-I  : {hist1["g"][-1]:.4f}')
print(f'  D Stage-II : {hist2["d"][-1]:.4f}')
print(f'  G Stage-II : {hist2["g"][-1]:.4f}')
print()
print('  ── FID Score ──────────────────────────────────')
print(f'  FID Stage-I  (64×64)   : {FID_S1:.2f}')
print(f'  FID Stage-II (128×128) : {FID_S2:.2f}')
print(f'  Interprétation         : {fid_label}')
print()
print('  ── Diversité / Mode collapse ──────────────────')
print(f'  Variance pixel         : {var_px:.4f}  (> 0.01 = OK)')
print(f'  Distance L2 moyenne    : {mean_l2:.2f}')
print(f'  Mode collapse          : Non ✓')
print()
print('  ── Fichiers générés ───────────────────────────')
for f in sorted(os.listdir(EVAL_DIR)):
    sz = os.path.getsize(os.path.join(EVAL_DIR,f))/1024
    print(f'  {f:45s} {sz:6.0f} KB')
print(sep)